In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [3]:
file_path = '../data/UserBehavior.csv'

In [4]:
df = pd.read_csv(file_path,nrows=None,names=['user_id','item_id','category_id','behavior_type','timestamp'])

In [5]:
print('数据形状（行数，列数）：',df.shape)
print(df.head())

数据形状（行数，列数）： (100150807, 5)
   user_id  item_id  category_id behavior_type   timestamp
0        1  2268318      2520377            pv  1511544070
1        1  2333346      2520771            pv  1511561733
2        1  2576651       149192            pv  1511572885
3        1  3830808      4181361            pv  1511593493
4        1  4365585      2520377            pv  1511596146


In [6]:
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("deep")
print(f'pandas版本：{pd.__version__}')
print(f'numpy版本：{np.__version__}')
print('库导入完成')

pandas版本：3.0.6
numpy版本：2.5.4
库导入完成


In [7]:
df.info()
print(df.describe())
print(df['behavior_type'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 100150807 entries, 0 to 100150806
Data columns (total 5 columns):
 #   Column         Dtype
---  ------         -----
 0   user_id        int64
 1   item_id        int64
 2   category_id    int64
 3   behavior_type  str  
 4   timestamp      int64
dtypes: int64(4), str(1)
memory usage: 3.7 GB
            user_id       item_id   category_id     timestamp
count  1.001508e+08  1.001508e+08  1.001508e+08  1.001508e+08
mean   5.069431e+05  2.579775e+06  2.696380e+06  1.511951e+09
std    2.940605e+05  1.488056e+06  1.463155e+06  5.528006e+06
min    1.000000e+00  1.000000e+00  8.000000e+01 -2.134949e+09
25%    2.524290e+05  1.295225e+06  1.320293e+06  1.511762e+09
50%    5.040150e+05  2.580735e+06  2.671397e+06  1.511965e+09
75%    7.609490e+05  3.862042e+06  4.145813e+06  1.512179e+09
max    1.018011e+06  5.163070e+06  5.162429e+06  2.122867e+09
behavior_type
pv      89716264
cart     5530446
fav      2888258
buy      2015839
Name: count, dtype: int64


In [10]:
missing = df.isna().sum()
duplicate_count = df.duplicated().sum()
print(missing)
print(duplicate_count)
if duplicate_count > 0:
    df.drop_duplicates(keep = 'first', inplace = True)
else:
    print('无重复数据')


user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64
0
无重复数据


In [11]:
df['date_time'] = pd.to_datetime(df['timestamp'],unit = 's')
df['date'] = df['date_time'].dt.date
df['hour'] = df['date_time'].dt.hour
df['weekday'] = df['date_time'].dt.weekday
weekday_map = {0:'周一',1:'周二',2:'周三',3:'周四',4:'周五',5:'周六',6:'周日'}
df['weekday_cn'] = df['weekday'].map(weekday_map)
df.head()

,user_id,item_id,category_id,behavior_type,timestamp,date_time,date,hour,weekday,weekday_cn
0,1,2268318,2520377,pv,1511544070,2017-11-24 17:21:10,2017-11-24,17,4,周五
1,1,2333346,2520771,pv,1511561733,2017-11-24 22:15:33,2017-11-24,22,4,周五
2,1,2576651,149192,pv,1511572885,2017-11-25 01:21:25,2017-11-25,1,5,周六
3,1,3830808,4181361,pv,1511593493,2017-11-25 07:04:53,2017-11-25,7,5,周六
4,1,4365585,2520377,pv,1511596146,2017-11-25 07:49:06,2017-11-25,7,5,周六


In [12]:
valid_behavior = ['pv','fav','cart','buy']
invalid_behavior = df[~df['behavior_type'].isin(valid_behavior)]
print(f'异常行为数量{len(invalid_behavior)}')
if len(invalid_behavior) > 0:
    df = df[df['behavior_type'].isin(valid_behavior)]
    print(f'已清除异常值，剩余{len(df)}条数据')
else:
    print('无异常行为')

异常行为数量0
无异常行为


In [17]:
start_date = pd.Timestamp('2017-11-25 00:00:00')
end_date = pd.Timestamp('2017-12-04 00:00:00')
out_of_range = df[(df['date_time'] < start_date) | (df['date_time'] > end_date)]
print(f'超出时间范围的记录数：{len(out_of_range)},占比为:{round(len(out_of_range)/len(df)*100,2)}%')
df = df[(df['date_time'] <= end_date) & (df['date_time'] >= start_date)]
print(f'过滤后剩余{len(df)}条记录')
daily_count = df.groupby('date').size().sort_index()
daily_count

超出时间范围的记录数：1236274,占比为:1.23%
过滤后剩余98914484条记录


date
2017-11-25    10511597
2017-11-26    10571039
2017-11-27    10013455
2017-11-28     9884185
2017-11-29    10319060
2017-11-30    10541695
2017-12-01    11171505
2017-12-02    13940942
2017-12-03    11961006
dtype: int64

In [21]:
all_users = df['user_id'].unique()
print(f'总用户数：{len(all_users)}')
np.random.seed(42)
sample_size = 50000
sampled_user = np.random.choice(all_users,size = sample_size,replace = False)
print(f'抽样用户数{len(sampled_user)}')
df_sample = df[df['user_id'].isin(sampled_user).copy()]
print(f'抽样后记录数{len(df_sample)}')
print(f'抽样比例：{round(len(df_sample)/len(df)*100,2)}%')
original_dist = df['behavior_type'].value_counts(normalize = True)*100
sample_dist = df_sample['behavior_type'].value_counts(normalize = True)*100
compare_df = pd.DataFrame({
    '原始数据（%）':original_dist.round(2),
    '抽样数据（%）':sample_dist.round(2),
    '差值（%）':(original_dist - sample_dist).round(2)
})
compare_df

总用户数：987984
抽样用户数50000
抽样后记录数5025843
抽样比例：5.08%


,原始数据（%）,抽样数据（%）,差值（%）
behavior_type,,,
pv,89.57,89.58,-0.01
cart,5.53,5.52,0.01
fav,2.88,2.89,-0.01
buy,2.02,2.01,0.01


In [23]:
output_path = '../data/sample_data.csv'
df_sample.to_csv(output_path,index=False,encoding='utf-8')
print(f"数据已保存至: {output_path}")
print(f"文件大小: 约 {len(df_sample)*50/1024/1024:.1f} MB")
print(f"数据行数: {len(df_sample):,}")
print(f"数据列数: {df_sample.shape[1]}")
print(f"包含字段: {list(df_sample.columns)}")

数据已保存至: ../data/sample_data.csv
文件大小: 约 239.7 MB
数据行数: 5,025,843
数据列数: 10
包含字段: ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp', 'date_time', 'date', 'hour', 'weekday', 'weekday_cn']


In [25]:
print(f"原始数据量: {len(df):,} 条")
print(f"清洗取样后数据量: {len(df_sample):,} 条")
print(f"用户数: {df_sample['user_id'].nunique():,}")
print(f"商品数: {df_sample['item_id'].nunique():,}")
print(f"品类数: {df_sample['category_id'].nunique():,}")
print(f"时间范围: {df_sample['date'].min()} ~ {df_sample['date'].max()}")
print(f"缺失值: {df_sample.isnull().sum().sum()} 个")
print(f"重复值: {df_sample.duplicated().sum()} 条")

原始数据量: 98,914,484 条
清洗取样后数据量: 5,025,843 条
用户数: 50,000
商品数: 1,106,777
品类数: 7,438
时间范围: 2017-11-25 ~ 2017-12-03
缺失值: 0 个
重复值: 0 条
